In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("AmesHousing.csv")
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Lot Frontage       490
Alley             2732
Mas Vnr Type      1775
Mas Vnr Area        23
Bsmt Qual           80
Bsmt Cond           80
Bsmt Exposure       83
BsmtFin Type 1      80
BsmtFin SF 1         1
BsmtFin Type 2      81
BsmtFin SF 2         1
Bsmt Unf SF          1
Total Bsmt SF        1
Electrical           1
Bsmt Full Bath       2
Bsmt Half Bath       2
Fireplace Qu      1422
Garage Type        157
Garage Yr Blt      159
Garage Finish      159
Garage Cars          1
Garage Area          1
Garage Qual        159
Garage Cond        159
Pool QC           2917
Fence             2358
Misc Feature      2824
dtype: int64

In [4]:
null_col_df.dtypes

Lot Frontage      float64
Alley              object
Mas Vnr Type       object
Mas Vnr Area      float64
Bsmt Qual          object
Bsmt Cond          object
Bsmt Exposure      object
BsmtFin Type 1     object
BsmtFin SF 1      float64
BsmtFin Type 2     object
BsmtFin SF 2      float64
Bsmt Unf SF       float64
Total Bsmt SF     float64
Electrical         object
Bsmt Full Bath    float64
Bsmt Half Bath    float64
Fireplace Qu       object
Garage Type        object
Garage Yr Blt     float64
Garage Finish      object
Garage Cars       float64
Garage Area       float64
Garage Qual        object
Garage Cond        object
Pool QC            object
Fence              object
Misc Feature       object
dtype: object

In [5]:
values = {
    # "Lot Frontage": imputation,
    "Alley": "No Alley Access",
    "Mas Vnr Type": "None",
    # "Mas Vnr Area": imputation,
    "Bsmt Qual": "No Basement",
    "Bsmt Cond": "No Basement",
    "Bsmt Exposure" : "No Basement",
    "BsmtFin Type 1" : "No Basement",
    # "BsmtFin SF 1" : imputation,
    "BsmtFin Type 2" : "No Basement",
    # "BsmtFin SF 2" : imputation,
    # "Bsmt Unf SF" : imputation,
    # "Total Bsmt SF" : imputation,
    # "Electrical" : might actually be missing,
    # "Bsmt Full Bath" : imputation,
    # "Bsmt Half Bath" : imputation,
    "Fireplace Qu" : "No Fireplace",
    "Garage Type" : "No Garage",
    # "Garage Yr Blt" : might actually be missing,
    "Garage Finish" : "No Garage",
    # "Garage Cars" : imputation,
    # "Garage Area" : imputation,
    "Garage Qual" : "No Garage",
    "Garage Cond" : "No Garage",
    "Pool QC" : "No Pool",
    "Fence" : "No Fence",
    "Misc Feature" : "None"
}

In [6]:
inconsistency1 = (df['Mas Vnr Type'].isna()) & (df['Mas Vnr Area'] > 0) # if Type is NA AND Area > 0
inconsistency2 = (~df['Mas Vnr Type'].isna()) & (df['Mas Vnr Area'] == 0) # if Type is not NA AND Area == 0

In [7]:
df[df['Mas Vnr Area'] < 0] # area should be nonnegative

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [8]:
df[inconsistency1][['Mas Vnr Type', 'Mas Vnr Area']]

,Mas Vnr Type,Mas Vnr Area
363,NaN,344.0
403,NaN,312.0
441,NaN,285.0
1861,NaN,1.0
1913,NaN,1.0
2003,NaN,1.0
2528,NaN,288.0


In [9]:
df[inconsistency2][['Mas Vnr Type', 'Mas Vnr Area']]

,Mas Vnr Type,Mas Vnr Area
1640,BrkFace,0.0
1740,BrkFace,0.0
1785,Stone,0.0


In [10]:
df.loc[inconsistency1, 'Mas Vnr Type'] = 'Unknown'
df.loc[inconsistency2, 'Mas Vnr Type'] = 'None'

In [11]:
bsmt_cat_var = ['Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2']
bsmt_num_var = ['BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath']
bsmt_vars = bsmt_cat_var + bsmt_num_var
bsmt_row_nas = df[bsmt_cat_var].isna().sum(axis=1)
inconsistency3 = (bsmt_row_nas < len(bsmt_cat_var)) & (bsmt_row_nas > 0) # the bsmt categorical features contradict each other
inconsistency4 = (df[bsmt_cat_var].isna().sum(axis=1) == len(bsmt_cat_var)) & (df[bsmt_num_var].sum(axis=1) > 0) # bsmt numerical features contradict bsmt categorical features

In [12]:
df[df[bsmt_num_var].sum(axis=1) < 0] # all numeric features should be nonnegative

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [13]:
df[inconsistency3][bsmt_vars]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
66,Gd,TA,NaN,Unf,Unf,0.0,0.0,1595.0,1595.0,0.0,0.0
444,Gd,TA,No,GLQ,NaN,1124.0,479.0,1603.0,3206.0,1.0,0.0
1796,Gd,TA,NaN,Unf,Unf,0.0,0.0,725.0,725.0,0.0,0.0
2779,Gd,TA,NaN,Unf,Unf,0.0,0.0,936.0,936.0,0.0,0.0


In [14]:
df[inconsistency4][bsmt_vars]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath


In [15]:
df.loc[inconsistency3, bsmt_cat_var] = df.loc[inconsistency3, bsmt_cat_var].fillna('Unknown')

In [16]:
garage_cat_var = ['Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond']
garage_num_var = ['Garage Yr Blt', 'Garage Cars', 'Garage Area']
garage_vars = garage_cat_var + garage_num_var
garage_row_nas = df[garage_cat_var].isna().sum(axis=1)
inconsistency5 = (garage_row_nas < len(garage_cat_var)) & (garage_row_nas > 0) # the garage categorical features contradict each other
inconsistency6 = (df[garage_cat_var].isna().sum(axis=1) == len(garage_cat_var)) & (df[garage_num_var].sum(axis=1) > 0) # garage numerical features contradict garage categorical features

In [17]:
df[df[garage_num_var].sum(axis=1) < 0] # all numeric columns should be nonnegative

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [18]:
df[inconsistency5][garage_vars]

,Garage Type,Garage Finish,Garage Qual,Garage Cond,Garage Yr Blt,Garage Cars,Garage Area
1356,Detchd,NaN,NaN,NaN,NaN,1.0,360.0
2236,Detchd,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df.loc[inconsistency5, garage_cat_var] = df.loc[inconsistency5, garage_cat_var].fillna('Unknown')

In [20]:
df[df['Garage Type'] == 'Detchd'][garage_num_var].median()

Garage Yr Blt    1962.0
Garage Cars         2.0
Garage Area       400.0
dtype: float64

In [21]:
df.loc[inconsistency5, 'Garage Yr Blt'] = 1962
df.loc[inconsistency5, 'Garage Cars'] = df.loc[inconsistency5, 'Garage Cars'].fillna(2)
df.loc[inconsistency5, 'Garage Area'] = df.loc[inconsistency5, 'Garage Area'].fillna(400)

In [22]:
df[inconsistency6][garage_vars]

,Garage Type,Garage Finish,Garage Qual,Garage Cond,Garage Yr Blt,Garage Cars,Garage Area


In [23]:
df[(df['Fireplaces'] > 0) & (df['Fireplace Qu'].isna())]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [24]:
df[(df['Pool Area'] > 0) & (df['Pool QC'].isna())]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [25]:
df[(df['Misc Val'] > 0) & (df['Misc Feature'].isna())]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [26]:
bsmt_total_check = df['BsmtFin SF 1'] + df['BsmtFin SF 2'] + df['Bsmt Unf SF'] == df['Total Bsmt SF']

In [27]:
df[bsmt_total_check][bsmt_vars]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
0,TA,Gd,Gd,BLQ,Unf,639.0,0.0,441.0,1080.0,1.0,0.0
1,TA,TA,No,Rec,LwQ,468.0,144.0,270.0,882.0,0.0,0.0
2,TA,TA,No,ALQ,Unf,923.0,0.0,406.0,1329.0,0.0,0.0
3,TA,TA,No,ALQ,Unf,1065.0,0.0,1045.0,2110.0,1.0,0.0
4,Gd,TA,No,GLQ,Unf,791.0,0.0,137.0,928.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2925,TA,TA,Av,GLQ,Unf,819.0,0.0,184.0,1003.0,1.0,0.0
2926,Gd,TA,Av,BLQ,ALQ,301.0,324.0,239.0,864.0,1.0,0.0
2927,Gd,TA,Av,GLQ,Unf,337.0,0.0,575.0,912.0,0.0,1.0
2928,Gd,TA,Av,ALQ,LwQ,1071.0,123.0,195.0,1389.0,1.0,0.0


In [28]:
df[~bsmt_total_check][bsmt_vars]

,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath
1341,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
live_area_check = df['1st Flr SF'] + df['2nd Flr SF'] + df['Low Qual Fin SF'] == df['Gr Liv Area']

In [30]:
df[live_area_check][['1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area']]

,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area
0,1656,0,0,1656
1,896,0,0,896
2,1329,0,0,1329
3,2110,0,0,2110
4,928,701,0,1629
...,...,...,...,...
2925,1003,0,0,1003
2926,902,0,0,902
2927,970,0,0,970
2928,1389,0,0,1389


In [31]:
garage_consist_check1 = (df['Garage Cars'] > 0) & (df['Garage Area'] > 0)
garage_consist_check2 = (df['Garage Cars'] == 0) & (df['Garage Area'] == 0)
df[~garage_consist_check1][['Garage Cars', 'Garage Area']]

,Garage Cars,Garage Area
27,0.0,0.0
119,0.0,0.0
125,0.0,0.0
129,0.0,0.0
130,0.0,0.0
...,...,...
2913,0.0,0.0
2916,0.0,0.0
2918,0.0,0.0
2919,0.0,0.0


In [32]:
df[garage_consist_check2][['Garage Cars', 'Garage Area']]

,Garage Cars,Garage Area
27,0.0,0.0
119,0.0,0.0
125,0.0,0.0
129,0.0,0.0
130,0.0,0.0
...,...,...
2913,0.0,0.0
2916,0.0,0.0
2918,0.0,0.0
2919,0.0,0.0


In [33]:
(df[~garage_consist_check1][['Garage Cars', 'Garage Area']] != df[garage_consist_check2][['Garage Cars', 'Garage Area']]).sum()

Garage Cars    0
Garage Area    0
dtype: int64

In [34]:
temp = df.dtypes.reset_index()
print(temp[0].value_counts())
temp = temp[temp[0] != 'object']
numeric_cols = temp['index'].unique()
categorical_cols = [col for col in df.columns if col not in numeric_cols]
df[df[numeric_cols].sum(axis=1) < 0]

0
object     43
int64      28
float64    11
Name: count, dtype: int64


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice


In [35]:
year_consist_check1 = (df['Year Remod/Add'] > df['Year Built']) | (df['Year Remod/Add'] == df['Year Built'])
index1 = df[~year_consist_check1].index
df[~year_consist_check1][['Year Remod/Add', 'Year Built']]

,Year Remod/Add,Year Built
850,2001,2002


In [36]:
year_consist_check2 = df['Yr Sold'] >= df['Year Built']
index2 = df[~year_consist_check2].index
df[~year_consist_check2][['Yr Sold', 'Year Built']]

,Yr Sold,Year Built
2180,2007,2008


In [37]:
year_consist_check3 = df['Yr Sold'] >= df['Garage Yr Blt']
index3 = df[~year_consist_check3 & ~df['Garage Yr Blt'].isna()].index
df[~year_consist_check3 & ~df['Garage Yr Blt'].isna()][['Yr Sold', 'Garage Yr Blt']]

,Yr Sold,Garage Yr Blt
2180,2007,2008.0
2260,2007,2207.0


In [38]:
df.loc[index1,['Year Remod/Add', 'Year Built']] = np.nan
df.loc[index2, ['Yr Sold', 'Year Built']] = np.nan
df.loc[index3, ['Yr Sold', 'Garage Yr Blt']] = np.nan

In [39]:
df.loc[list(index1) + list(index2) + list(index3), ['Year Remod/Add', 'Yr Sold', 'Year Built', 'Garage Yr Blt']]

,Year Remod/Add,Yr Sold,Year Built,Garage Yr Blt
850,NaN,2009.0,NaN,2002.0
2180,2009.0,NaN,NaN,NaN
2180,2009.0,NaN,NaN,NaN
2260,2007.0,NaN,2006.0,NaN


In [40]:
index = df[df['Garage Yr Blt'].isna()].index
df.loc[df['Year Built'].isna(), 'Year Built'] = df['Year Built'].median()
df.loc[index, 'Garage Yr Blt'] = df.loc[index, 'Year Built']
df.loc[df['Year Remod/Add'].isna(), 'Year Remod/Add'] = df['Year Remod/Add'].median()
df.loc[df['Yr Sold'].isna(), 'Yr Sold'] = df['Yr Sold'].median()

In [41]:
df.fillna(value=values, inplace=True)

In [42]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Lot Frontage      490
Mas Vnr Area       23
BsmtFin SF 1        1
BsmtFin SF 2        1
Bsmt Unf SF         1
Total Bsmt SF       1
Electrical          1
Bsmt Full Bath      2
Bsmt Half Bath      2
dtype: int64

In [43]:
df.loc[df['BsmtFin SF 1'].isna(), 'BsmtFin SF 1'] = df['BsmtFin SF 1'].median()
df.loc[df['BsmtFin SF 2'].isna(), 'BsmtFin SF 2'] = df['BsmtFin SF 2'].median()
df.loc[df['Bsmt Unf SF'].isna(), 'Bsmt Unf SF'] = df['Bsmt Unf SF'].median()
index = df.loc[df['Total Bsmt SF'].isna(), 'Total Bsmt SF'].index
df.loc[index, 'Total Bsmt SF'] = (df['BsmtFin SF 1'] + df['BsmtFin SF 2'] + df['Bsmt Unf SF']).loc[index]

In [44]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Lot Frontage      490
Mas Vnr Area       23
Electrical          1
Bsmt Full Bath      2
Bsmt Half Bath      2
dtype: int64

In [45]:
# group-based imputation
df['Lot Frontage'] = df['Lot Frontage'].fillna(
    df.groupby(['Neighborhood', 'Lot Config'])['Lot Frontage'].transform('median')
)
# global imputation for the 7 rows still missing
df['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].median())

In [46]:
df[df['Mas Vnr Area'].isna()][['Mas Vnr Type', 'Mas Vnr Area']]

,Mas Vnr Type,Mas Vnr Area
55,None,NaN
484,None,NaN
517,None,NaN
538,None,NaN
867,None,NaN
1095,None,NaN
1119,None,NaN
1122,None,NaN
1127,None,NaN
1184,None,NaN


In [47]:
df.loc[df['Mas Vnr Area'].isna(), 'Mas Vnr Area'] = 0

In [48]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Electrical        1
Bsmt Full Bath    2
Bsmt Half Bath    2
dtype: int64

In [51]:
df['Electrical'].mode()[0]

'SBrkr'

In [55]:
df.loc[df['Electrical'].isna(), 'Electrical'] = df['Electrical'].mode()[0]
df.loc[df['Bsmt Full Bath'].isna(), 'Bsmt Full Bath'] = df['Bsmt Full Bath'].mode()[0]
df.loc[df['Bsmt Half Bath'].isna(), 'Bsmt Half Bath'] = df['Bsmt Half Bath'].mode()[0]

In [56]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

Series([], dtype: float64)